In [1]:
# imports

import os
import gradio as gr

from openai import OpenAI
from dotenv import load_dotenv

/Users/aj/Desktop/Learnings/Projects/Conversational-AI-Assistant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load environment variables
load_dotenv(override=True)

# get the API key from the environment variable
openai_api_key = os.getenv("OPENAI_API_KEY")

# initialize OpenAI client
client = OpenAI(api_key=openai_api_key)


In [3]:
MODEL = "gpt-5.6-luna"
MESSAGE = [
    {"role": "system", "content": "You are helpful assistant"}
]


In [6]:
def chat(message, history):
    system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
    the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
    For example, if the customer says 'I'm looking to buy a hat', \
    you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
    Encourage the customer to buy hats if they are unsure what to get."

    system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
    but remind the customer to look at hats!"

    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role":"system", "content": relevant_system_message}] + history + [{"role":"user", "content":message}]
    stream = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response
    return response
    

In [7]:
gr.ChatInterface(fn=chat, title="Conversational AI Assistant", description="Ask anything you want").launch( )

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
